# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## stg_question
- 질문 원문(question_text)을 제공하는 사전(dictionary) 역할이 필요하기에 생성
- 이후 분석할 '어떤 질문에 반응이 좋은가' 에 대한 기준 테이블이 됨

## 타입 확인
- created_at TIMESTAMP(KST 변환 대상)

In [ ]:
%%bigquery df_q_types --project your-gcp-project
SELECT column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'polls_question'
ORDER BY ordinal_position;

In [ ]:
df_q_types

## 품질 및 텍스트 확인

- `distinct_text` = 3,903 ← 총 5,025개인데 고유 텍스트는 3,903개. 즉 1,122개가 중복 텍스트
- 같은 질문이 여러 id로 존재

In [ ]:
%%bigquery df_q_qc --project your-gcp-project
SELECT
  COUNT(*)                              AS total_questions,
  COUNT(*) - COUNT(DISTINCT id)         AS dup_id,          -- 0이어야 (grain)
  COUNTIF(question_text IS NULL)        AS null_text,        -- 질문 원문 결측
  COUNT(DISTINCT question_text)         AS distinct_text,    -- ★ 텍스트 종류 수
  MIN(DATE(DATETIME(created_at,'Asia/Seoul'))) AS first_q,
  MAX(DATE(DATETIME(created_at,'Asia/Seoul'))) AS last_q
FROM `your-gcp-project.raw.polls_question`;

In [ ]:
df_q_qc

## 실제 질문 미리보기
- 외모·성격·관계·재미 등 다양한 칭찬형 질문

In [ ]:
%%bigquery df_q_sample --project your-gcp-project
SELECT id, question_text,
       DATETIME(created_at,'Asia/Seoul') AS created_at
FROM `your-gcp-project.raw.polls_question`
ORDER BY id
LIMIT 30;

In [ ]:
df_q_sample

## 중복 원인 확인
- 1위 'vote' : 테스트 데이터로 추측 가능, 분석 대상에서 제외할 것
- "눈이 제일 큰 사람은?"(3개), "가장 잘생긴 사람은?"(2개, 4월→5월) 등 : 같은 질문을 시즌마다 다시 낸 케이스
- `first_seen`이랑 `last_seen`이 다른 것이 그 증거(예: 4/01 처음 → 5/15 재등장)
> 인기 질문을 재활용하는 서비스 운영 패턴

In [ ]:
%%bigquery df_dup_text --project your-gcp-project
-- 같은 question_text가 여러 id로 존재하는 케이스 확인
SELECT
  question_text,
  COUNT(*)              AS id_count,       -- 몇 개 id로 존재하나
  MIN(id)               AS first_id,
  MAX(id)               AS last_id,
  MIN(DATE(DATETIME(created_at,'Asia/Seoul'))) AS first_seen,
  MAX(DATE(DATETIME(created_at,'Asia/Seoul'))) AS last_seen
FROM `your-gcp-project.raw.polls_question`
GROUP BY question_text
HAVING COUNT(*) > 1
ORDER BY id_count DESC
LIMIT 20;

In [ ]:
df_dup_text

## 더미 후보 탐색
- 물음표 없어도 정상적인 질문 많음
- 진짜 더미는 vote 하나 뿐

In [ ]:
%%bigquery df_dummy_check --project your-gcp-project
-- 질문답지 않은 텍스트 탐색 (더미 후보)
SELECT
  question_text,
  COUNT(*) AS cnt
FROM `your-gcp-project.raw.polls_question`
WHERE question_text NOT LIKE '%?%'          -- 물음표 없는 것 (질문은 대부분 ?로 끝남)
   OR LENGTH(TRIM(question_text)) <= 3      -- 너무 짧은 것
GROUP BY question_text
ORDER BY cnt DESC
LIMIT 30;

In [ ]:
df_dummy_check

## Stage 생성

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_question` AS
WITH src AS (
  SELECT id, question_text, created_at
  FROM `your-gcp-project.raw.polls_question`
)
SELECT
  id                                 AS question_id,        -- 조인키 (vote.question_id ↔ 여기)
  question_text,                                            -- 원본 텍스트
  TRIM(question_text)                AS question_text_norm,  -- 그룹핑용 정규화 (공백만 제거)
  -- 더미 질문 플래그: 'vote'와 빈값만 제외, 나머지는 전부 유효
  (TRIM(question_text) != 'vote'
     AND TRIM(question_text) != '')  AS is_valid_question,
  DATETIME(created_at, 'Asia/Seoul') AS created_at
FROM src;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                              AS total_questions,   -- 5,025
  COUNTIF(is_valid_question)            AS valid_questions,   -- 5,025 - 56 = 4,969 예상
  COUNTIF(NOT is_valid_question)        AS dummy_questions,   -- 56 (vote) 예상
  COUNT(DISTINCT question_text_norm)    AS distinct_norm      -- 정규화 후 고유 텍스트 수
FROM `your-gcp-project.stage.stg_question`;